## Load Data

In [ ]:
data_dicts = {}
for id in [759,863,503,30,579,193,890,887,880,264,296,891]:
  # fetch dataset
  df = fetch_ucirepo(id=id)

  # data (as pandas dataframes)
  X = df.data.features
  X = X.loc[:, ~X.columns.duplicated()]
  X_cols=X.columns.to_list()

  y = df.data.targets.iloc[:,0]

  # Calculate imbalance ratio
  y_counts = Counter(y)
  majority_class = max(y_counts.values())
  minority_class = min(y_counts.values())
  imbalance_ratio = majority_class / minority_class if minority_class != 0 else float('inf')

  # metadata
  print(id)
  if id in mappings:
    y=y.map(mappings[id])
  print(y.unique())

  # Create lists
  var_info=df.variables
  cat_cols = var_info[((var_info['type'] == 'Categorical') | (var_info['type'] == 'Binary')) & (var_info['role'] != 'Target') &(var_info['role'] != 'ID')  ]['name'].tolist()

  cat_cols=[col for col in cat_cols if col in X_cols]
  num_cols=list(set(X_cols)-set(cat_cols))
  print(cat_cols)
  print(num_cols)
  X_filtered=X[cat_cols + num_cols]
  print(X_filtered.columns.to_list())
  # Create dictionary to save in pickle file
  data_dict = {
        'name' : df.metadata.name,
        'nrows' : X.shape[0],
        'ncat' : len(cat_cols),
        'nnum' : len(num_cols),
        'imbalance_ratio' : imbalance_ratio,
        'X': X_filtered,
        'y': y.values,
        'cat_cols': cat_cols,
        'num_cols': num_cols
    }

  if id == 296:
      # Create dictionary to save in pickle file
      data_dict = {
            'name' : df.metadata.name,
            'nrows' : X.shape[0],
            'ncat' : len(cat_cols),
            'nnum' : len(num_cols),
            'imbalance_ratio' : imbalance_ratio,
            'X': X_filtered.head(40000),
            'y': y.head(40000).values,
            'cat_cols': cat_cols,
            'num_cols': num_cols
        }

  data_dicts[id]=data_dict

### Run Baseline

In [ ]:
for i,(df_name,d) in enumerate(data_dicts.items()):
  print('-------------------------------------')
  print(df_name)
  X,y=d['X'],d['y']
  cat_col,num_col=d['cat_cols'],d['num_cols']

  X_train, X_test,y_train, y_test = train_test_split(X,y, train_size=0.8,random_state=1,stratify=y)
  print(X_train.shape)

  output_dir=f"/content/drive/MyDrive/Experiment/VAE/D{df_name}/"
  if not os.path.exists(output_dir):
      os.makedirs(output_dir)
  # Set the current working directory
  os.chdir(output_dir)
  # Verify the change
  print("Current working directory:", os.getcwd())

  models, model_names = setup_models(X_train,cat_col,num_col)
  print(train_evaluate_models(models, model_names, X_train, X_test, y_train, y_test,True,'baseline.csv'))

##ACVAE

In [ ]:
import pickle
# Specify the path to the pickle file
pickle_file_path = '/content/drive/MyDrive/Experiment/VAE/data_dicts.pkl'


with open(pickle_file_path, "rb") as input_file:
  data_dicts = pickle.load(input_file)

In [ ]:
pd.DataFrame(data_dicts).T

,name,nrows,ncat,nnum,imbalance_ratio,X,y,cat_cols,num_cols
759,Glioma Grading Clinical and Mutation Features,839,22,1,1.383523,Gender Race IDH1 ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[Gender, Race, IDH1, TP53, ATRX, PTEN, EGFR, C...",[Age_at_diagnosis]
863,Maternal Health Risk,1014,0,6,1.492647,Age HeartRate BS SystolicBP Diasto...,"[2, 2, 2, 2, 0, 2, 1, 2, 1, 2, 0, 1, 0, 1, 1, ...",[],"[Age, HeartRate, BS, SystolicBP, DiastolicBP, ..."
503,Hepatitis C Virus (HCV) for Egyptian patients,1385,9,19,1.090361,Gender Fever Nausea/Vomting Headache ...,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ...","[Gender, Fever, Nausea/Vomting, Headache , Dia...","[ALT 1, RNA EOT, RNA 4, ALT 24, RNA Base, BMI,..."
30,Contraceptive Method Choice,1473,7,2,1.888889,wife_edu husband_edu wife_religion wi...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[wife_edu, husband_edu, wife_religion, wife_wo...","[wife_age, num_children]"
579,Myocardial infarction complications,1700,94,17,9.0,SEX INF_ANAM STENOK_AN FK_STENOK IBS...,"[0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[SEX, INF_ANAM, STENOK_AN, FK_STENOK, IBS_POST...","[D_AD_KBRIG, S_AD_KBRIG, NA_BLOOD, K_BLOOD, NA..."
193,Cardiotocography,2126,0,21,10.924528,LB ASTV MLTV DL UC MSTV Max...,"[9, 6, 6, 6, 2, 8, 8, 9, 9, 9, 10, 10, 6, 2, 6...",[],"[LB, ASTV, MLTV, DL, UC, MSTV, Max, ALTV, Nmax..."
890,AIDS Clinical Trials Group Study 175,2139,11,12,3.105566,hemo homo drugs oprior z30 zprior ...,"[0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, ...","[hemo, homo, drugs, oprior, z30, zprior, gende...","[trt, wtkg, time, strat, cd40, karnof, cd820, ..."
887,National Health and Nutrition Health Survey 20...,2278,0,7,5.258242,LBXGLU BMXBMI LBXIN PAQ605 RIAGENDR ...,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, ...",[],"[LBXGLU, BMXBMI, LBXIN, PAQ605, RIAGENDR, DIQ0..."
880,SUPPORT2,9105,10,32,2.135331,sex dzgroup dz...,"[0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, ...","[sex, dzgroup, dzclass, edu, income, race, ca,...","[glucose, alb, crea, wblc, urine, charges, hda..."
264,EEG Eye State,14980,0,14,1.228172,F7 P7 T8 O1 ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[],"[F7, P7, T8, O1, O2, F4, AF4, F3, FC5, FC6, F8..."


In [ ]:
# Example usage
device = 'cuda' if torch.cuda.is_available() else 'cpu'

batch_size=64
embedding_dim=256
compress_dims=(128, 128)
decompress_dims=(128, 128)
early_stop_thresh=100

epochs=1000

In [ ]:
for i in [759,863,503,30,579,264,296,891,193,890,887,880]:
  df_name=i
  d=data_dicts[i]
  print('-------------------------------------')
  print(df_name)
  X,y=d['X'],d['y']
  cat_col,num_col=d['cat_cols'],d['num_cols']
  n_classes=len(np.unique(y))

  output_dir=f"/content/drive/MyDrive/Experiment/VAE/D{df_name}/"
  if not os.path.exists(output_dir):
      os.makedirs(output_dir)
  # Set the current working directory
  os.chdir(output_dir)
  # Verify the change
  print("Current working directory:", os.getcwd())

  X_train, X_test,y_train, y_test = train_test_split(X.fillna(0.0),y, train_size=0.8,random_state=1,stratify=y)
  print(X_train.shape,n_classes)

  transformer = DataTransformer()
  transformer.fit(X_train,cat_col)
  pickle.dump(transformer, open('transformer.pkl', 'wb'))
  X_train_enc = transformer.transform(X_train)
  pickle.dump(X_train_enc, open('X_train_enc.pkl', 'wb'))
  n_samples,data_dim=X_train_enc.shape

  real_dataset = TensorDataset(torch.from_numpy(X_train_enc),torch.from_numpy(y_train).long())
  loader = DataLoader(real_dataset, batch_size=batch_size, shuffle=True)

  acvae = acvae(data_dim,compress_dims, decompress_dims, embedding_dim, n_classes, device, use_aux=True)
  # Create a list of the model's attributes
  export_list = [data_dim,compress_dims, decompress_dims, embedding_dim, n_classes, device, True]
  # Export to pickle file
  pickle.dump(export_list, open('model_params.pkl', 'wb'))

  print("Model parameters exported successfully.")
  acvae.train(loader,epochs=epochs, early_stop_thresh=early_stop_thresh,transformer=transformer)
  acvae.save_model("ctencoder.pth", "ctdecoder.pth")

In [ ]:
for i in [891]:#,887,880]759,863,503,30,579,264,296,891,193,
  df_name=i
  d=data_dicts[i]
  print('-------------------------------------')
  print(df_name)
  X,y=d['X'],d['y']
  cat_col,num_col=d['cat_cols'],d['num_cols']
  n_classes=len(np.unique(y))

  output_dir=f"/content/drive/MyDrive/Experiment/VAE/D{df_name}/"
  if not os.path.exists(output_dir):
      os.makedirs(output_dir)
  # Set the current working directory
  os.chdir(output_dir)
  # Verify the change
  print("Current working directory:", os.getcwd())

  X_train, X_test,y_train, y_test = train_test_split(X,y, train_size=0.8,random_state=1,stratify=y)
  transformer = pickle.load(open('transformer.pkl', 'rb'))
  X_train_enc = pickle.load(open('X_train_enc.pkl', 'rb'))
  loaded_params = pickle.load(open('model_params.pkl', 'rb'))
  # Create a new instance of acvae using the loaded parameters
  acvae = acvae(*loaded_params)
  acvae.load_model("ctencoder.pth", "ctdecoder.pth")

  X_train, X_test,y_train, y_test = train_test_split(X,y, train_size=0.8,random_state=1,stratify=y)

  models, model_names = setup_models(X_train,cat_col,num_col,True)
  X_bal,y_bal=acvae.generate_fake_data(X_train_enc,y_train,transformer,False)
  print(train_evaluate_models(models, model_names, X_bal ,X_test, y_bal, y_test,filepath="ACVAE.csv"))

  X_bal,y_bal=acvae.generate_fake_data(X_train_enc,y_train,transformer,False,EditedECDNN())
  print(train_evaluate_models(models, model_names, X_bal ,X_test, y_bal, y_test,filepath="ACVAECDNN.csv"))